# PATSTAT — stage 0: the dump's zip parts to parquet

PATSTAT Global 2023 Autumn arrives as 65 zip parts, one CSV each (`/project/jevans/PATSTAT/unzipped_data/`,
~280 GB uncompressed). Every other notebook here reads the tables through DuckDB, which cannot read a
zip, so this stage streams each part once to a zstd parquet under `PATSTAT/raw/<tls>/partMM.parquet`
with the column types from the edition's own DDL (`ps_common.TABLES`). The dump itself is never written to.

Normally this runs as the SLURM array `jobs/PATSTAT/load.sbatch` (one task per part, ~21 parts for the
11 registered tables). This notebook is the serial fallback and the verification: it loads whatever is
still missing, then checks every table's row count against the official `RowCount_2023b_result.txt`.

## Tables registered (and why)
| table | rows | used for |
|---|---|---|
| tls201_appln | 128 M | the application universe, filing year, family, granted flag |
| tls211_pat_publn | 152 M | publication -> application routing, grant year |
| tls212_citation | 508 M | the citations (publication -> publication / application) |
| tls209_appln_ipc, tls224_appln_cpc | 341 M, 383 M | classification: IPC main, CPC list, CPC subclass pairs (atypicality) |
| tls230_appln_techn_field | 151 M | WIPO technology field -> the five sectors (hit cohorts) |
| tls207_pers_appln, tls206_person | 351 M, 90 M | inventors / applicants, countries, applicant sector |
| tls202_appln_title, tls204_appln_prior, tls228_docdb_fam_citn | | titles, priorities, family-level citations (not read by the metrics; loaded for later use) |

In [ ]:
import os, sys, time
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
print(ps.EDITION); print('dump:', ps.DUMP); print('raw :', ps.RAW)
jobs = ps.load_jobs()
print(f'{len(jobs)} zip parts over {len(ps.TABLES)} tables')

In [ ]:
%%time
# Loads the parts not yet present, serially. Idle when the SLURM array has already done the work.
todo = [(t, p) for t, p in jobs if not os.path.exists(f"{ps.raw_dir(t)}/part{p[-6:-4]}.parquet")]
print(f'{len(todo)} parts to load')
for t, p in todo:
    ps.load_table(t, p)

In [ ]:
ps.summary()
for tls in ps.TABLES:
    if ps.raw_complete(tls):
        assert ps.raw_rows(tls) == ps.TABLES[tls]['rows'], f'{tls}: row count differs from the official count'
print('\nevery complete table matches the official row count')